# 10. Kadane's Algorithm and Subarray Patterns

**Maximum subarray sum** is the most asked array question. Kadane solves it in O(n) with one idea: at each element either **extend** the previous subarray or **start fresh**.

**Learning order:** IDEA → KADANE → WITH INDICES → CIRCULAR → ABSOLUTE SUM → STOCK LINK → INTERVIEW THINKING

## 1. The idea

`cur` = best sum of a subarray **ending here**. If the previous `cur` is negative it only drags us down, so start over:

`cur = max(x, cur + x)` and `best = max(best, cur)`.

Real life: the best consecutive stretch of days in a trading history, or the most profitable consecutive months.

In [1]:
def max_subarray(nums):
    best = cur = nums[0]
    for x in nums[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best

print(max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4]), max_subarray([-3, -1, -2]))   # 6 -1

6 -1


## 2. Return the subarray itself

Track where the current run **starts**. When `cur` restarts at `x`, set `start = i`. When `best` improves, store `(start, i)`.

In [2]:
def max_subarray_with_indices(nums):
    best = cur = nums[0]
    start = best_l = best_r = 0
    for i in range(1, len(nums)):
        if cur < 0:                       # previous run only hurts: restart
            cur, start = nums[i], i
        else:
            cur += nums[i]
        if cur > best:
            best, best_l, best_r = cur, start, i
    return best, best_l, best_r

print(max_subarray_with_indices([-2, 1, -3, 4, -1, 2, 1, -5, 4]))   # (6, 3, 6)

(6, 3, 6)


## 3. Maximum Sum Circular Subarray

In a circular array the best subarray is either:
1. **Normal** (does not wrap): ordinary Kadane.
2. **Wrapping:** it is `total - (minimum subarray)`: remove the worst middle part.

Answer = `max(normal, total - min_sub)`. **Edge case:** if all numbers are negative, `total - min_sub` would be 0 (empty subarray), so return the normal result.

In [3]:
def max_subarray_circular(nums):
    total = sum(nums)
    best_max = cur_max = best_min = cur_min = nums[0]
    for x in nums[1:]:
        cur_max = max(x, cur_max + x)
        best_max = max(best_max, cur_max)
        cur_min = min(x, cur_min + x)
        best_min = min(best_min, cur_min)
    if best_max < 0:
        return best_max
    return max(best_max, total - best_min)

print(max_subarray_circular([1, -2, 3, -2]), max_subarray_circular([5, -3, 5]), max_subarray_circular([-3, -2, -3]))   # 3 10 -2

3 10 -2


## 4. Maximum Absolute Sum of Any Subarray

The largest **absolute** sum is either the maximum subarray sum or the negative of the minimum subarray sum.

In [4]:
def max_absolute_sum(nums):
    cur_max = cur_min = best = 0
    for x in nums:
        cur_max = max(0, cur_max + x)
        cur_min = min(0, cur_min + x)
        best = max(best, cur_max, -cur_min)
    return best

print(max_absolute_sum([1, -3, 2, 3, -4]), max_absolute_sum([2, -5, 1, -4, 3, -2]))   # 5 8

5 8


## 5. Link: best time to buy and sell stock

Max profit with one transaction = max subarray sum of the **daily price differences**. Equivalent to tracking the lowest price so far.

In [5]:
def max_profit(prices):
    diffs = [prices[i] - prices[i - 1] for i in range(1, len(prices))]
    best = cur = 0
    for d in diffs:
        cur = max(0, cur + d)
        best = max(best, cur)
    return best

print(max_profit([7, 1, 5, 3, 6, 4]), max_profit([7, 6, 4, 3, 1]))   # 5 0

5 0


## 6. Interview thinking

**Signals:** "maximum/minimum sum of a contiguous subarray", "best stretch", "circular", "absolute".

**Say:** "`cur` is the best sum ending at this index. If it is negative I drop it. O(n) time, O(1) space." Mention the **prefix sum** view: `best = max(prefix[i] - min(prefix[j])) for j < i`.

**Common mistakes:** initializing `best = 0` (wrong when all numbers are negative), circular case with all negatives, confusing subarray (contiguous) with subsequence.

## Quick Revision

- Kadane: `cur = max(x, cur + x)`, `best = max(best, cur)`. Initialize with `nums[0]`.
- Indices: restart `start` when `cur < 0`.
- Circular: `max(kadane_max, total - kadane_min)`, guard all-negative.
- Absolute: max(`max_sub`, `-min_sub`).